# train.csv の英訳冒頭脱落点検

## 目的
- `train.csv` の `transliteration` と `translation` を突き合わせ、英訳側で定型句や冒頭要素が落ちていそうな行を機械的に洗い出す。
- まずは高精度のヒューリスティックに限定し、`karum Kaniš` や `Say to ...` のような定型の脱落候補を一覧化する。

## このノートブックでやること
1. `train.csv` を読み込む
2. 転写側の定型句と、英訳側で期待する表現の対応表を定義する
3. `transliteration` に定型句があるのに、`translation` に対応表現が見当たらない行を候補として抽出する

## 注意
- これは自動修正用ではなく、**要確認候補の抽出用**。
- 意訳や言い換えは誤検出することがある。
- OCR/転記ゆれの確認は、必要に応じて PDF / published_texts / 近傍の並行例で手動確認する。

In [ ]:
from pathlib import Path
import re

import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    """`train.csv` が見つかるまで親ディレクトリを遡る。"""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data/kaggle/deep-past-initiative-machine-translation/train.csv").exists():
            return candidate
    raise FileNotFoundError("repo root を特定できませんでした。")


repo_root = find_repo_root()
train_path = repo_root / "data/kaggle/deep-past-initiative-machine-translation/train.csv"
train = pd.read_csv(train_path)

print(f"repo_root: {repo_root}")
print(f"train rows: {len(train):,}")
train[["oare_id", "transliteration", "translation"]].head(2)

## 対応表の初版

以下の対応表は、**転写側の定型トリガー**と**英訳側で最低限見えてほしい表現**を対応付けた初版です。  
判定は単純な正規表現ベースで、まずは再現性を優先しています。

In [ ]:
rules = [
    {
        "rule_id": "intro_karum_kanish",
        "rule_name": "karum Kaniš 導入",
        "translit_regex": r"um-ma\s+kà-ru-um\s+kà-ni-iš-ma",
        "translation_regex": r"\bkarum\s+kan(?:e|i)sh\b",
        "expected_translation": "Thus karum Kanesh / Kanish",
        "note": "`Thus Kaniš` だけになっている行を拾う。",
    },
    {
        "rule_id": "intro_say_to",
        "rule_name": "宛先の導入",
        "translit_regex": r"a-na\s+.+?\s+q[ií]-bi[0-9]?-ma",
        "translation_regex": r"\b(?:say\s+to|to\s+.+?,\s*say)\b",
        "expected_translation": "Say to ... / To ..., say",
        "note": "冒頭の `a-na ... qí-bi-ma` が訳で丸ごと抜けるケースを想定。",
    },
    {
        "rule_id": "intro_hear_our_letter",
        "rule_name": "書簡受領の定型句",
        "translit_regex": r"ṭup-p[ìi]-ni\s+ta-ša-me-a-ni",
        "translation_regex": r"\b(?:as soon as|when)\s+you\s+hear\s+our\s+letter\b",
        "expected_translation": "As soon as you hear our letter",
        "note": "条件節ごと落ちていないかを見る。",
    },
    {
        "rule_id": "dagger_of_assur",
        "rule_name": "Aššur の剣",
        "translit_regex": r"GÍR\s+ša\s+a-šur",
        "translation_regex": r"\bdagger\s+of\s+a[šs]+ur\b|\ba[šs]+ur'?s\s+dagger\b",
        "expected_translation": "dagger of Aššur / Aššur's dagger",
        "note": "重要語が英訳から落ちていないかを見る。",
    },
    {
        "rule_id": "we_hear",
        "rule_name": "nišammea の導入",
        "translit_regex": r"ni-ša-me",
        "translation_regex": r"\bwe\s+hear\b|\bit\s+has\s+been\s+reported\b",
        "expected_translation": "We hear ...",
        "note": "報告導入が丸ごと落ちた行を拾う。",
    },
    {
        "rule_id": "must_not_delay",
        "rule_name": "遅延禁止",
        "translit_regex": r"lá\s+i-sa-ḫu-ur|u4-ma-kál\s+lá\s+i-sà-ḫu-ur",
        "translation_regex": r"\b(?:must\s+not\s+(?:be\s+)?delayed|do\s+not\s+delay|not\s+suffer\s+a\s+delay)\b",
        "expected_translation": "must not be delayed / do not delay",
        "note": "書簡末尾の命令句が欠けていないかを見る。",
    },
]

rules_df = pd.DataFrame(rules)[["rule_id", "rule_name", "translit_regex", "expected_translation", "note"]]
rules_df

In [ ]:
def find_translation_omission_candidates(df: pd.DataFrame, rules: list[dict]) -> pd.DataFrame:
    """転写に定型句があるのに、英訳に期待表現が見当たらない行を抽出する。"""
    rows = []
    for row in df.itertuples(index=False):
        transliteration = row.transliteration if isinstance(row.transliteration, str) else ""
        translation = row.translation if isinstance(row.translation, str) else ""
        for rule in rules:
            if not re.search(rule["translit_regex"], transliteration, flags=re.IGNORECASE):
                continue
            if re.search(rule["translation_regex"], translation, flags=re.IGNORECASE):
                continue
            rows.append(
                {
                    "oare_id": row.oare_id,
                    "rule_id": rule["rule_id"],
                    "rule_name": rule["rule_name"],
                    "expected_translation": rule["expected_translation"],
                    "transliteration": transliteration,
                    "translation": translation,
                }
            )
    return pd.DataFrame(rows)


candidates = find_translation_omission_candidates(train, rules)
print(f"候補件数: {len(candidates):,}")
candidates.groupby(["rule_id", "rule_name"]).size().sort_values(ascending=False).rename("count").reset_index()

In [ ]:
# まずは今回の論点に直結する `karum Kaniš` の脱落候補を確認する。
karum_candidates = candidates.loc[candidates["rule_id"] == "intro_karum_kanish", [
    "oare_id",
    "rule_name",
    "transliteration",
    "translation",
]]

karum_candidates.head(20)

## 次の拡張候補
- `karum Kaniš` のような**語単位脱落**だけでなく、人数・地名列の縮退も検出する
- `translation` 側にある綴り揺れ (`Assur` / `Aššur`, `Kanesh` / `Kanish`) を許容する正規表現を増やす
- 候補一覧を `oare_id` / 出版物単位で review 用 CSV に落とす
- PDF スクリーンショットで確認済みの行に `review_status` を付けて再利用する